# ML Zoomcamp 2026 — Homework 3 (Classification)

Dataset: `course_lead_scoring_2026.csv` (pinned 2026 release)

Submit: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03

Deadline: Mon 12 Oct 2026, 23:00

---

## Big picture (what are we doing?)

We have leads (people who showed interest in a course).  
Each row is one lead. The target column is **`converted`**:

- `1` = they signed up
- `0` = they did not

We will:
1. Clean the data
2. Explore which features seem useful (correlation + mutual information)
3. Train a **logistic regression** model to predict conversion
4. Check accuracy, drop features one-by-one, and tune regularization (`C`)

**Your job:** fill in the `YOUR CODE` / `YOUR ANSWER` cells.  
When you have an answer, tell me (e.g. "Q1 = technology") and I will validate it.

## Setup

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression

## Load data + prepare missing values

**Why?** Models and many stats hate empty cells.

Homework rule:
- categorical (text) missing → replace with `'NA'`
- numerical (numbers) missing → replace with `0.0`

In [3]:
df = pd.read_csv('course_lead_scoring_2026.csv')

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

# Fill missing values as required
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


---
## Question 1 — Mode of `industry`

**Simple idea:** *Mode* = the value that appears most often.

Hint: `df['industry'].value_counts()` or `df['industry'].mode()`

Options: `NA` / `technology` / `healthcare` / `retail`

In [6]:
# YOUR CODE — find the most frequent value of industry
mode=df['industry'].mode()
print(f"mode of industry is '{mode[0]}'")

mode of industry is 'technology'


**YOUR ANSWER Q1:** _(write here after you run the cell)_

---
## Question 2 — Biggest correlation (numerical features)

**Simple idea:** Correlation says how two numbers move together.

- Close to `+1` → when one goes up, the other tends to go up
- Close to `0` → little linear relationship
- Close to `-1` → they move in opposite directions

Hint: `df[numerical].corr()` then look only at the pairs listed in the options.

Options:
- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- `annual_income` and `interaction_count`

In [8]:
# YOUR CODE — correlation matrix for numerical columns
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


**YOUR ANSWER Q2:** intreaction_count and lead_score _(write the pair with the biggest correlation)_

---
## Split the data (required exact split)

**Simple idea:** We hide some data so we can honestly check the model later.

- **train** → learn patterns
- **val** → tune / pick settings
- **test** → final check (not used until later; homework uses train + val)

Sizes with this split: 60% train / 20% val / 20% test.

Also: remove `converted` from the feature frames (keep it as `y_...`).

In [9]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

---
## Question 3 — Biggest mutual information (categorical)

**Simple idea:** Mutual information (MI) asks:  
*"If I know this category, how much does that help me guess `converted`?"*

- Higher MI → more useful categorical feature
- Use **training set only**
- Round to 2 decimals: `round(score, 2)`

Hint:
```python
mutual_info_score(y_train, df_train['industry'])
```

Options: `industry` / `location` / `lead_source` / `employment_status`

In [15]:
# YOUR CODE — MI between converted (y_train) and each categorical column
print(round(mutual_info_score(y_train,df_train['industry']),2))
print(round(mutual_info_score(y_train,df_train['location']),2))
print(round(mutual_info_score(y_train,df_train['lead_source']),2))
print(round(mutual_info_score(y_train,df_train['employment_status']),2))

0.0
0.0
0.03
0.02


**YOUR ANSWER Q3:** employment_status_(feature with biggest MI)_

---
## Question 4 — Logistic regression accuracy

**Simple ideas:**

1. **One-hot encoding:** turn categories into 0/1 columns  
   Example: `industry=technology` → a column that is 1 only for technology rows.

2. **Logistic regression:** predicts probability of conversion (0–1), then we turn that into a class.

3. **Accuracy:** fraction of validation predictions that match the true label.

Required model settings:
```python
LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
```

Round validation accuracy to **2** decimals.

Options: `0.55` / `0.65` / `0.75` / `0.85`

Hint (common course pattern):
```python
from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
```

In [23]:
# YOUR CODE — one-hot encode, fit logistic regression, accuracy on val
from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_val)
round(accuracy_score(y_val, y_pred), 2)

0.65

**YOUR ANSWER Q4:** _(rounded accuracy)_

---
## Question 5 — Feature elimination (least useful feature)

**Simple idea:** Train the full model, then remove one feature at a time and see how much accuracy changes.

- Difference = `original_accuracy - accuracy_without_feature`
- Smallest difference (closest to zero, can be negative) → least useful among the options

Do this for:
- `lead_source`
- `number_of_courses_viewed`
- `interaction_count`

Use the same model settings as Q4. Compare using the **unrounded** original accuracy.

In [24]:
# YOUR CODE — drop each feature, retrain, compare accuracy differences
features = categorical + numerical

def train_and_score(feature_list):
    dv = DictVectorizer(sparse=False)

    train_dicts = df_train[feature_list].to_dict(orient='records')
    X_train = dv.fit_transform(train_dicts)

    val_dicts = df_val[feature_list].to_dict(orient='records')
    X_val = dv.transform(val_dicts)

    model = LogisticRegression(
        solver='liblinear', C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

# original accuracy (do NOT round for the difference)
original_acc = train_and_score(features)
print('original:', original_acc)

candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for f in candidates:
    acc_without = train_and_score([x for x in features if x != f])
    diff = original_acc - acc_without
    print(f, 'acc=', acc_without, 'diff=', diff)

original: 0.645
lead_source acc= 0.642 diff= 0.0030000000000000027
number_of_courses_viewed acc= 0.643 diff= 0.0020000000000000018
interaction_count acc= 0.601 diff= 0.04400000000000004


**YOUR ANSWER Q5:** _(feature with smallest difference)_

---
## Question 6 — Tune regularization `C`

**Simple idea:** `C` controls how strongly the model is pushed to stay "simple".

- Smaller `C` → stronger regularization (simpler model)
- Larger `C` → weaker regularization (model can fit training data more tightly)

Try: `[0.000001, 0.00001, 0.0001, 0.001]`  
Round validation accuracy to **3** decimals.  
If there is a tie, pick the **smallest** `C`.

Use all features, same setup as Q4 (only change `C`).

In [25]:
# YOUR CODE — loop over C values and report val accuracy (3 decimals)
features = categorical + numerical

dv = DictVectorizer(sparse=False)
train_dicts = df_train[features].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[features].to_dict(orient='records')
X_val = dv.transform(val_dicts)

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(
        solver='liblinear', C=C, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train)
    acc = accuracy_score(y_val, model.predict(X_val))
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


**YOUR ANSWER Q6:** _(best C)_

---
## Answer sheet (fill after each question)

| Q | Your answer |
|---|-------------|
| 1 |  |
| 2 |  |
| 3 |  |
| 4 |  |
| 5 |  |
| 6 |  |

When ready, paste your answers here in chat and I will validate them.